### Demo — Inferencia en vídeo real urbano

En esta demo aplicamos el modelo v2 (YOLOv8s entrenado en VisDrone) sobre un vídeo real de tráfico urbano grabado desde perspectiva aérea. El objetivo es demostrar la viabilidad del sistema en un entorno real, detectando coches, personas y otros vehículos frame a frame y reconstruyendo el vídeo con las predicciones dibujadas encima.

Este tipo de demo representa el caso de uso final del proyecto — un sistema de vigilancia urbana inteligente capaz de procesar vídeo en tiempo real desde cámaras elevadas.

In [1]:
!yt-dlp -o '../data/video_demo.mp4' 'https://youtu.be/bfc2wsX29zk?si=n4hnf9pBC0FI4Je4'

[youtube] Extracting URL: https://youtu.be/bfc2wsX29zk?si=n4hnf9pBC0FI4Je4
[youtube] bfc2wsX29zk: Downloading webpage
[youtube] bfc2wsX29zk: Downloading android vr player API JSON
[info] bfc2wsX29zk: Downloading 1 format(s): 315+251
[download] Destination: ../data/video_demo.mp4.f315.webm
[download] 100% of  123.61MiB in 00:01:00 at 2.03MiB/s0;33m00:000m
[download] Destination: ../data/video_demo.mp4.f251.webm
[download] 100% of   21.17KiB in 00:00:02 at 9.18KiB/s0;33m00:000m
[Merger] Merging formats into "../data/video_demo.mp4.webm"
Deleting original file ../data/video_demo.mp4.f315.webm (pass -k to keep)
Deleting original file ../data/video_demo.mp4.f251.webm (pass -k to keep)


#### Tarea 1 — Inferencia sobre vídeo de tráfico: solo coches

El primer vídeo de prueba muestra una carretera de alta velocidad grabada desde perspectiva aérea. Es el caso más sencillo porque solo aparece una clase — coches — que además es la que mejor detecta nuestro modelo con un mAP@0.5 de 0.724.

El objetivo es verificar que el pipeline de inferencia sobre vídeo funciona correctamente antes de pasar a escenas más complejas con múltiples clases.

En este caso no aplicamos SAHI ya que los coches en una carretera son objetos relativamente grandes y bien visibles — SAHI está pensado para escenas con objetos muy pequeños y alta densidad, como las intersecciones urbanas de VisDrone. Aquí la inferencia estándar es suficiente.

In [4]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# Rutas
DATA_DIR = Path('../data')
video_path = DATA_DIR / 'video_demo.mp4.webm'

# Cargar modelo v2
model = YOLO('../notebooks/runs/notebooks/runs/models/visdrone_yolov8s_640/weights/best.pt')

print(f"Vídeo existe: {video_path.exists()}")
print(f"Modelo cargado correctamente")

Vídeo existe: True
Modelo cargado correctamente


In [5]:
cap = cv2.VideoCapture(str(video_path))

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"FPS: {fps}")
print(f"Total frames: {total_frames}")
print(f"Resolución: {width}x{height}")
print(f"Duración: {total_frames/fps:.1f} segundos")

cap.release()

FPS: 59.94006309148265
Total frames: 2836
Resolución: 3840x2160
Duración: 47.3 segundos


In [6]:
CLASES_VISDRONE = {0: 'pedestrian', 1: 'people', 2: 'bicycle', 3: 'car', 
                   4: 'van', 5: 'truck', 6: 'tricycle', 7: 'awning-tricycle', 
                   8: 'bus', 9: 'motor'}

cap = cv2.VideoCapture(str(video_path))
output_path = DATA_DIR / 'video_demo_output.mp4'

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(str(output_path), fourcc, 30, (3840, 2160))

frame_count = 0
processed = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Procesar 1 de cada 2 frames
    if frame_count % 2 == 0:
        results = model.predict(frame, conf=0.3, verbose=False)
        
        for box in results[0].boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            label = f"{CLASES_VISDRONE[cls]} {conf:.2f}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
            cv2.putText(frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        out.write(frame)
        processed += 1
        
        if processed % 50 == 0:
            print(f"Frames procesados: {processed}/{total_frames//2}")
    
    frame_count += 1

cap.release()
out.release()
print("Vídeo guardado en", output_path)

Frames procesados: 50/1418
Frames procesados: 100/1418
Frames procesados: 150/1418
Frames procesados: 200/1418
Frames procesados: 250/1418
Frames procesados: 300/1418
Frames procesados: 350/1418
Frames procesados: 400/1418
Frames procesados: 450/1418
Frames procesados: 500/1418
Frames procesados: 550/1418
Frames procesados: 600/1418
Frames procesados: 650/1418
Frames procesados: 700/1418
Frames procesados: 750/1418
Frames procesados: 800/1418
Frames procesados: 850/1418
Frames procesados: 900/1418
Frames procesados: 950/1418
Frames procesados: 1000/1418
Frames procesados: 1050/1418
Frames procesados: 1100/1418
Frames procesados: 1150/1418
Frames procesados: 1200/1418
Frames procesados: 1250/1418
Frames procesados: 1300/1418
Frames procesados: 1350/1418
Frames procesados: 1400/1418
Vídeo guardado en ../data/video_demo_output.mp4


In [8]:
!ffmpeg -i '../data/video_demo_output.mp4' -vcodec libx264 '../data/video_demo_h264.mp4' -y

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [9]:
from IPython.display import Video
output_h264 = DATA_DIR / 'video_demo_h264.mp4'
Video(str(output_h264), width=800)

El modelo v2 detecta los coches con gran precisión en este vídeo de carretera. Las predicciones son estables frame a frame y los bounding boxes siguen correctamente los vehículos en movimiento, incluso a alta velocidad.

Este resultado es coherente con el mAP@0.5 de 0.724 obtenido para la clase car en el test set — es la clase mejor representada en VisDrone y la que el modelo aprende con más solidez.

En este escenario la inferencia estándar es completamente suficiente — los coches son objetos grandes y bien visibles desde perspectiva aérea, por lo que SAHI no aportaría mejora significativa.

#### Tarea 2 — Inferencia sobre vídeo de tráfico: múltiples clases

El segundo vídeo añade complejidad al escenario — aparecen camiones, furgonetas y distintos tipos de vehículos además de coches. Esto nos permite evaluar cómo el modelo discrimina entre clases similares como car, van y truck, que visualmente son parecidas desde perspectiva aérea.

En este caso tampoco aplicamos SAHI ya que los vehículos siguen siendo objetos de tamaño considerable. El objetivo es ver si el modelo clasifica correctamente los distintos tipos de vehículos.

In [10]:
!yt-dlp -o '../data/video_demo2.%(ext)s' 'https://youtu.be/6PhY3SrJk2A?si=8UxRKU_h6vvssb3V'

[youtube] Extracting URL: https://youtu.be/6PhY3SrJk2A?si=8UxRKU_h6vvssb3V
[youtube] 6PhY3SrJk2A: Downloading webpage
[youtube] 6PhY3SrJk2A: Downloading android vr player API JSON
[info] 6PhY3SrJk2A: Downloading 1 format(s): 313+251
[download] Destination: ../data/video_demo2.f313.webm
[download] 100% of   29.69MiB in 00:00:15 at 1.93MiB/s0;33m00:000m
[download] Destination: ../data/video_demo2.f251.webm
[download] 100% of   11.58KiB in 00:00:00 at 51.25KiB/s;33mUnknown
[Merger] Merging formats into "../data/video_demo2.webm"
Deleting original file ../data/video_demo2.f251.webm (pass -k to keep)
Deleting original file ../data/video_demo2.f313.webm (pass -k to keep)


In [11]:
video_path2 = DATA_DIR / 'video_demo2.webm'

cap = cv2.VideoCapture(str(video_path2))

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"FPS: {fps}")
print(f"Total frames: {total_frames}")
print(f"Resolución: {width}x{height}")
print(f"Duración: {total_frames/fps:.1f} segundos")

cap.release()

FPS: 29.97002997002997
Total frames: 767
Resolución: 3840x2160
Duración: 25.6 segundos


In [12]:
output_path2 = DATA_DIR / 'video_demo2_output.mp4'

cap = cv2.VideoCapture(str(video_path2))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(str(output_path2), fourcc, 30, (3840, 2160))

frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    results = model.predict(frame, conf=0.3, verbose=False)
    
    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        label = f"{CLASES_VISDRONE[cls]} {conf:.2f}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
        cv2.putText(frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    
    out.write(frame)
    frame_count += 1
    
    if frame_count % 50 == 0:
        print(f"Frames procesados: {frame_count}/{total_frames}")

cap.release()
out.release()
print("Vídeo guardado")

Frames procesados: 50/767
Frames procesados: 100/767
Frames procesados: 150/767
Frames procesados: 200/767
Frames procesados: 250/767
Frames procesados: 300/767
Frames procesados: 350/767
Frames procesados: 400/767
Frames procesados: 450/767
Frames procesados: 500/767
Frames procesados: 550/767
Frames procesados: 600/767
Frames procesados: 650/767
Frames procesados: 700/767
Frames procesados: 750/767
Vídeo guardado


In [13]:
!ffmpeg -i '../data/video_demo2_output.mp4' -vcodec libx264 '../data/video_demo2_h264.mp4' -y

output_h264_2 = DATA_DIR / 'video_demo2_h264.mp4'
Video(str(output_h264_2), width=800)

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [ ]:
output_path2 = DATA_DIR / 'video_demo2_output.mp4'

cap = cv2.VideoCapture(str(video_path2))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(str(output_path2), fourcc, 30, (3840, 2160))

frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    results = model.predict(frame, conf=0.3, verbose=False)
    
    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        label = f"{CLASES_VISDRONE[cls]} {conf:.2f}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
        cv2.putText(frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    
    out.write(frame)
    frame_count += 1
    
    if frame_count % 50 == 0:
        print(f"Frames procesados: {frame_count}/{total_frames}")

cap.release()
out.release()
print("Vídeo guardado")

Frames procesados: 50/767
Frames procesados: 100/767
Frames procesados: 150/767
Frames procesados: 200/767
Frames procesados: 250/767
Frames procesados: 300/767
Frames procesados: 350/767
Frames procesados: 400/767
Frames procesados: 450/767
Frames procesados: 500/767
Frames procesados: 550/767
Frames procesados: 600/767
Frames procesados: 650/767
Frames procesados: 700/767
Frames procesados: 750/767
Vídeo guardado


El modelo v2 demuestra buena capacidad para discriminar entre clases similares — coches, furgonetas y camiones se detectan correctamente con confianzas altas. El rendimiento es especialmente bueno en zonas de aparcamiento con alta densidad de vehículos.

Los coches en movimiento en la carretera se detectan bien aunque con confianza ligeramente inferior debido al motion blur. Los camiones, al ser una clase menos representada en VisDrone, obtienen confianzas más bajas pero siguen siendo detectados correctamente.

Este vídeo confirma que el modelo generaliza bien a escenas aéreas urbanas reales más allá del dataset de test.

## Tarea 3 — Inferencia sobre vídeo urbano complejo: personas y vehículos

El tercer vídeo es el más desafiante — muestra una intersección urbana concurrida con peatones cruzando y múltiples tipos de vehículos circulando simultáneamente. Es el escenario más cercano al caso de uso real del proyecto.

A diferencia de los vídeos anteriores, aquí sí aplicaremos SAHI además de la inferencia estándar para comparar visualmente si mejora la detección de peatones, que son objetos pequeños y difíciles de detectar desde perspectiva aérea.

In [14]:
!yt-dlp -o '../data/video_demo3.%(ext)s' 'https://youtu.be/Bqhp2YU8XjU?si=FExN9g-Ko-1PPLbf'

[youtube] Extracting URL: https://youtu.be/Bqhp2YU8XjU?si=FExN9g-Ko-1PPLbf
[youtube] Bqhp2YU8XjU: Downloading webpage
[youtube] Bqhp2YU8XjU: Downloading android vr player API JSON
[info] Bqhp2YU8XjU: Downloading 1 format(s): 137+251
[download] Destination: ../data/video_demo3.f137.mp4
[download] 100% of   23.17MiB in 00:00:06 at 3.74MiB/s0;33m00:000m
[download] Destination: ../data/video_demo3.f251.webm
[download] 100% of  762.99KiB in 00:00:00 at 1.44MiB/s0;33m00:000m
[Merger] Merging formats into "../data/video_demo3.mkv"
Deleting original file ../data/video_demo3.f251.webm (pass -k to keep)
Deleting original file ../data/video_demo3.f137.mp4 (pass -k to keep)


In [15]:
import glob
video_path3 = list(DATA_DIR.glob('video_demo3.*'))[0]
print(f"Archivo: {video_path3.name}")

cap = cv2.VideoCapture(str(video_path3))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"FPS: {fps}")
print(f"Total frames: {total_frames}")
print(f"Resolución: {width}x{height}")
print(f"Duración: {total_frames/fps:.1f} segundos")
cap.release()

Archivo: video_demo3.mkv
FPS: 29.97002997002997
Total frames: 1405
Resolución: 1920x1080
Duración: 46.9 segundos


In [16]:
output_path3 = DATA_DIR / 'video_demo3_output.mp4'

cap = cv2.VideoCapture(str(video_path3))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(str(output_path3), fourcc, 30, (1920, 1080))

frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    results = model.predict(frame, conf=0.3, verbose=False)
    
    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        label = f"{CLASES_VISDRONE[cls]} {conf:.2f}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    
    out.write(frame)
    frame_count += 1
    
    if frame_count % 50 == 0:
        print(f"Frames procesados: {frame_count}/{total_frames}")

cap.release()
out.release()
print("Vídeo guardado")

Frames procesados: 50/1405
Frames procesados: 100/1405
Frames procesados: 150/1405
Frames procesados: 200/1405
Frames procesados: 250/1405
Frames procesados: 300/1405
Frames procesados: 350/1405
Frames procesados: 400/1405
Frames procesados: 450/1405
Frames procesados: 500/1405
Frames procesados: 550/1405
Frames procesados: 600/1405
Frames procesados: 650/1405
Frames procesados: 700/1405
Frames procesados: 750/1405
Frames procesados: 800/1405
Frames procesados: 850/1405
Frames procesados: 900/1405
Frames procesados: 950/1405
Frames procesados: 1000/1405
Frames procesados: 1050/1405
Frames procesados: 1100/1405
Frames procesados: 1150/1405
Frames procesados: 1200/1405
Frames procesados: 1250/1405
Frames procesados: 1300/1405
Frames procesados: 1350/1405
Frames procesados: 1400/1405
Vídeo guardado


In [ ]:
import glob
video_path3 = list(DATA_DIR.glob('video_demo3.*'))[0]
print(f"Archivo: {video_path3.name}")

cap = cv2.VideoCapture(str(video_path3))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"FPS: {fps}")
print(f"Total frames: {total_frames}")
print(f"Resolución: {width}x{height}")
print(f"Duración: {total_frames/fps:.1f} segundos")
cap.release()

Archivo: video_demo3.mkv
FPS: 29.97002997002997
Total frames: 1405
Resolución: 1920x1080
Duración: 46.9 segundos


In [17]:
!ffmpeg -i '../data/video_demo3_output.mp4' -vcodec libx264 '../data/video_demo3_h264.mp4' -y

output_h264_3 = DATA_DIR / 'video_demo3_h264.mp4'
Video(str(output_h264_3), width=800)

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

El tercer vídeo es el más complejo y representativo del caso de uso real. Es una intersección urbana asiática con altísima densidad de tráfico, múltiples tipos de vehículos y peatones cruzando simultáneamente.

El modelo v2 demuestra una capacidad de detección notable en este escenario:

**Lo que funciona bien:**
- Los buses se detectan perfectamente con bounding box muy preciso
- Los coches y furgonetas en zonas más despejadas obtienen confianzas altas (0.73-0.87)
- Los camiones se clasifican correctamente distinguiéndolos de coches y furgonetas
- La clase motor destaca — el modelo detecta correctamente la gran cantidad de motocicletas características de esta escena urbana
- Los peatones aparecen detectados en pasos de cebra y aceras con confianzas entre 0.30 y 0.43, coherente con el mAP@0.5 de 0.278 obtenido en test

**Lo que es más difícil:**
- La zona de alta densidad con cientos de motos solapadas genera confianzas más bajas y algunos bounding boxes se solapan
- Los peatones en zonas muy concurridas no siempre son detectados individualmente por su pequeño tamaño y oclusión

Los tres vídeos demuestran que el modelo v2 generaliza correctamente a escenas urbanas reales muy distintas entre sí — carreteras de alta velocidad, intersecciones mixtas y cruces urbanos densos. Este tercer escenario es donde SAHI aportaría la mayor mejora en producción, reduciendo los fallos en objetos pequeños y zonas de alta densidad.